In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import os
import json
import requests
from bs4 import BeautifulSoup
import shutil
import time
import csv

In [2]:
ids= pd.read_csv(r'E:\ml-32m/links.csv')

In [3]:
# we are gonaa pull data from tmdb api using movie lens data set ids
# we will get movie trailers,psoters etc
ids.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [4]:
TMDB_API_KEY = "dcbbf062e91bfc7ee36c9aa743dbba09"
TMDB_BASE_URL = "https://api.themoviedb.org/3"
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"
EXTRA_BACKUP_FILE = "tmdb_movies_extra_backup.csv"

In [ ]:
def get_tmdb_extra_data(tmdb_id):
    params = {"api_key": TMDB_API_KEY}
    url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None
    data = response.json()

    # Images
    images_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/images"
    images_response = requests.get(images_url, params=params)
    images = images_response.json().get("posters", []) if images_response.status_code == 200 else []
    poster_urls = [TMDB_IMAGE_BASE_URL + img["file_path"] for img in images if img.get("file_path")]

    # Videos
    videos_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/videos"
    videos_response = requests.get(videos_url, params=params)
    videos = videos_response.json().get("results", []) if videos_response.status_code == 200 else []
    video_keys = [video["key"] for video in videos if video.get("key")]

    # Reviews
    reviews_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/reviews"
    reviews_response = requests.get(reviews_url, params=params)
    reviews = reviews_response.json().get("results", []) if reviews_response.status_code == 200 else []
    review_texts = [review["content"] for review in reviews if review.get("content")]

    # External IDs
    external_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/external_ids"
    external_response = requests.get(external_url, params=params)
    external_ids = external_response.json() if external_response.status_code == 200 else {}

    # Release dates (for certification)
    release_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/release_dates"
    release_response = requests.get(release_url, params=params)
    certification = None
    if release_response.status_code == 200:
        results = release_response.json().get("results", [])
        for entry in results:
            if entry.get("iso_3166_1") == "US":
                for rel in entry.get("release_dates", []):
                    if rel.get("certification"):
                        certification = rel["certification"]
                        break
                if certification:
                    break

    # Cast with images
    credits_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/credits"
    credits_response = requests.get(credits_url, params=params)
    cast_list = []
    if credits_response.status_code == 200:
        for member in credits_response.json().get("cast", []):
            cast_list.append({
                "name": member.get("name"),
                "profile_url": TMDB_IMAGE_BASE_URL + member["profile_path"] if member.get("profile_path") else None
            })

    result = {
        "movie_id": tmdb_id,
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "tagline": data.get("tagline"),
        "poster_urls": poster_urls,
        "video_keys": video_keys,
        "review_texts": review_texts,
        "external_ids": external_ids,
        "certification": certification,
        "cast": cast_list
    }
    return result

# Load backup if exists
if os.path.exists(EXTRA_BACKUP_FILE):
    extra_df = pd.read_csv(EXTRA_BACKUP_FILE,quoting=csv.QUOTE_ALL, engine="python", on_bad_lines='skip')
    done_ids = set(extra_df["movie_id"].astype(str))
    print(f"Resuming from backup, {len(done_ids)} movies already processed.")
else:
    extra_df = pd.DataFrame()
    done_ids = set()

results = []
tmdb_ids = ids["tmdbId"].dropna().astype(int).astype(str).tolist()
start_idx = len(done_ids)

for idx, tmdb_id in enumerate(tmdb_ids[start_idx:], start=start_idx):
    if tmdb_id in done_ids:
        continue
    extra_data = get_tmdb_extra_data(tmdb_id)
    if extra_data:
        results.append(extra_data)
    time.sleep(0.25)  # Respect TMDB rate limits
    # Save backup every 10 movies
    if (idx + 1) % 10 == 0 or (idx + 1) == len(tmdb_ids):
        temp_df = pd.DataFrame(results)
        extra_df = pd.concat([extra_df, temp_df], ignore_index=True)
        extra_df.to_csv(EXTRA_BACKUP_FILE, index=False)
        results = []
        actual_saved_count = len(set(extra_df["movie_id"].astype(str)))
        print(f"Saved backup at {actual_saved_count} movies.")

print("Done! All extra movie data processed and saved.")